# Practical 7 - Relevance, BM25, Boosting and Profiling

A hands-on relevance lab against Elasticsearch 9.x
(`http://127.0.0.1:9200`) using the official Python client and a 60-project
real-estate dataset.

The nine tasks:

1. Search for a common keyword.
2. Observe `_score` and compare documents.
3. Compare scores across different documents.
4. Inspect the score with the Explain API.
5. Boost project-name matches.
6. Assign different importance to different fields.
7. Implement `function_score`.
8. Compare results before and after boosting.
9. Investigate query execution with the Profile API.

The assignment builds `relevance_search()`: exact project names rank highest,
partial names lower, builder names moderately and locations as a smaller
contribution. The final section explains BM25 and shows the ranking changes.

**Prerequisites:** Elasticsearch running (Practical 1), `generate_data.py`
executed here, and the shared `es` kernel (see `README.md`).

## Setup - client, data and helpers

`projects.json` contains 60 documents and a controlled signal cluster for the
query **"palm beach"**:

| ID | Name | Builder | Locality | Signal |
|----|------|---------|----------|--------|
| PJ011 | Palm Beach | Palm Beach Developers | Palm Beach Road | exact name |
| PJ012 | Palm Beach Residences | Palm Beach Developers | Palm Beach Road | partial name |
| PJ013 | Palm Beach Enclave | Skyline Builders | Palm Beach Road | partial name |
| PJ014 | Royal Palm Grove | Palm Estates | Palm Beach Road | location |
| PJ015 | Crescent Bay | Palm Beach Developers | Nerul | builder |

PJ016 (`Skyline`) and PJ018 (`Crescent Bay Vista`, builder Skyline Builders)
isolate the name-vs-builder field importance experiment.

Helpers:

- `search()` runs a query and keeps a small source projection.
- `score_table()` prints rank / id / score / name / builder / locality and the
  signals each hit matched.
- `rank_map()` converts a response into `{project_id: rank}`.
- `explain_walk()` and `profile_types()` flatten the Explain and Profile trees.

In [1]:
import json
import re
from pathlib import Path

from elasticsearch import Elasticsearch
from elasticsearch.helpers import bulk

ES_URL = "http://127.0.0.1:9200"
INDEX = "projects"
es = Elasticsearch(ES_URL, request_timeout=60)

info = es.info()
print("Connected to Elasticsearch", info["version"]["number"])
print("cluster:", info["cluster_name"], "| node:", info["name"])

HERE = Path.cwd()
DATA = json.loads((HERE / "projects.json").read_text(encoding="utf-8"))
BY_ID = {p["project_id"]: p for p in DATA}
print("projects loaded:", len(DATA))

FIELDS = ["project_id", "name", "builder", "locality", "city", "description"]


def pretty(obj):
    return json.dumps(obj, indent=2, default=str)


def words(text):
    return re.findall(r"[a-z0-9]+", str(text).lower())


def phrase_in(text, phrase):
    return phrase in " ".join(words(text))


def signals(project, query):
    labels = []
    name = " ".join(words(project["name"]))
    if name == query:
        labels.append("exact-name")
    elif phrase_in(project["name"], query):
        labels.append("partial-name")
    if phrase_in(project["builder"], query):
        labels.append("builder")
    if phrase_in(project["locality"], query) or phrase_in(project["city"], query):
        labels.append("location")
    if phrase_in(project["description"], query):
        labels.append("description")
    return "+".join(labels) or "-"


def search(query, size=10, profile=False):
    return es.search(
        index=INDEX,
        query=query,
        size=size,
        source_includes=FIELDS,
        profile=profile,
    )


def hits_of(response):
    return response["hits"]["hits"]


def ids_of(response):
    return [hit["_id"] for hit in hits_of(response)]


def total_hits(response):
    return response["hits"]["total"]["value"]


def rank_map(response):
    return {hit["_id"]: rank for rank, hit in enumerate(hits_of(response), 1)}


def score_map(response):
    return {hit["_id"]: hit["_score"] for hit in hits_of(response)}


def scores(response):
    return [hit["_score"] for hit in hits_of(response)]


def score_table(response, limit=10, query_text=None):
    print(f"{'#':>2} {'id':<6} {'score':>9}  {'name':<26} {'builder':<24} "
          f"{'locality':<16} signals")
    print("-" * 118)
    for rank, hit in enumerate(hits_of(response)[:limit], 1):
        source = hit["_source"]
        label = signals(source, query_text) if query_text else "-"
        print(f"{rank:>2} {hit['_id']:<6} {hit['_score']:>9.4f}  "
              f"{source['name']:<26} {source['builder']:<24} "
              f"{source['locality']:<16} {label}")


def explain_walk(node, rows, depth=0, max_depth=4):
    description = node.get("description", "")
    if description.startswith(("idf", "tf,", "dl,", "avgdl", "boost")):
        rows.append((depth, node.get("value"), description))
    if depth >= max_depth:
        return
    for child in node.get("details", []):
        explain_walk(child, rows, depth + 1, max_depth)


def profile_types(node):
    found = [node.get("type", "?")]
    for child in node.get("children", []):
        found.extend(profile_types(child))
    return found

Connected to Elasticsearch 9.2.0
cluster: my-elasticsearch-cluster | node: my-node
projects loaded: 60


### Index settings and mappings

Ranking uses the default **BM25** similarity (`k1=1.2`, `b=0.75`). The mapping
keeps analyzed text for matching and keyword variants for exact values:

- `name` - `text` + `name.keyword` (lowercase/asciifolding normalizer)
- `builder`, `locality`, `city` - `text` + `keyword` sub-fields
- `description` - `text`
- popularity - `rating`, `reviews_count`, `views`, `featured`, `created_at`

`name.keyword` uses a normalizer so the exact-name clause is case-insensitive
without tokenizing, which is what lets the assignment put a true exact match
above a phrase match.

In [2]:
SETTINGS = {
    "analysis": {
        "normalizer": {
            "lowercase_normalizer": {
                "type": "custom",
                "filter": ["lowercase", "asciifolding"],
            }
        }
    }
}

MAPPING = {
    "properties": {
        "project_id": {"type": "keyword"},
        "name": {
            "type": "text",
            "fields": {
                "keyword": {
                    "type": "keyword",
                    "normalizer": "lowercase_normalizer",
                    "ignore_above": 256,
                }
            },
        },
        "description": {"type": "text"},
        "builder": {
            "type": "text",
            "fields": {"keyword": {"type": "keyword"}},
        },
        "locality": {
            "type": "text",
            "fields": {"keyword": {"type": "keyword"}},
        },
        "city": {
            "type": "text",
            "fields": {"keyword": {"type": "keyword"}},
        },
        "property_type": {"type": "keyword"},
        "status": {"type": "keyword"},
        "price_min": {"type": "integer"},
        "price_max": {"type": "integer"},
        "bhk": {"type": "keyword"},
        "amenities": {"type": "keyword"},
        "possession_date": {"type": "date"},
        "created_at": {"type": "date"},
        "rera_approved": {"type": "boolean"},
        "rating": {"type": "float"},
        "reviews_count": {"type": "integer"},
        "views": {"type": "integer"},
        "featured": {"type": "boolean"},
    }
}


def recreate(index, settings, mappings):
    if es.indices.exists(index=index):
        es.indices.delete(index=index)
    es.indices.create(index=index, settings=settings, mappings=mappings)


recreate(INDEX, SETTINGS, MAPPING)
bulk(
    es,
    [
        {"_index": INDEX, "_id": project["project_id"], "_source": project}
        for project in DATA
    ],
    refresh=True,
)

count = es.count(index=INDEX)["count"]
print("index:", INDEX, "| documents:", count)
print("similarity: BM25 defaults (k1=1.2, b=0.75)")
print("exact-name field: name.keyword with normalizer", SETTINGS["analysis"]["normalizer"])
assert count == len(DATA) == 60

index: projects | documents: 60
similarity: BM25 defaults (k1=1.2, b=0.75)
exact-name field: name.keyword with normalizer {'lowercase_normalizer': {'type': 'custom', 'filter': ['lowercase', 'asciifolding']}}


## Task 1 - Search for a Common Keyword

`palm` is a common keyword in this catalogue: it appears in project names, in
the builder **Palm Estates**, in the locality **Palm Beach Road** and in
descriptions. A `multi_match` over the five text fields returns every document
whose analyzed terms match any field.

BM25 is not "the document with the most mentions wins"; field length, term
frequency and inverse document frequency all matter, so several documents can
share a score even with different names.

In [3]:
common_query = {
    "multi_match": {
        "query": "palm",
        "fields": ["name", "builder", "locality", "city", "description"],
    }
}
common = search(common_query, size=10)
print("query:", pretty(common_query))
print("total matches:", total_hits(common))
print()
score_table(common, query_text="palm")

points = scores(common)
assert total_hits(common) >= 10
assert points == sorted(points, reverse=True)
assert all(point > 0 for point in points)
assert "PJ011" in ids_of(common)
assert "PJ003" in ids_of(common)
print()
print("top score:", round(points[0], 4), "| lowest shown:", round(points[-1], 4))

query: {
  "multi_match": {
    "query": "palm",
    "fields": [
      "name",
      "builder",
      "locality",
      "city",
      "description"
    ]
  }
}
total matches: 14

 # id         score  name                       builder                  locality         signals
----------------------------------------------------------------------------------------------------------------------
 1 PJ011     2.2261  Palm Beach                 Palm Beach Developers    Palm Beach Road  partial-name+builder+location+description
 2 PJ012     2.2261  Palm Beach Residences      Palm Beach Developers    Palm Beach Road  partial-name+builder+location+description
 3 PJ014     2.2261  Royal Palm Grove           Palm Estates             Palm Beach Road  partial-name+builder+location+description
 4 PJ010     2.0409  Palm Springs               Palm Estates             Kamothe          partial-name+builder+description
 5 PJ028     2.0409  Palm Residences            Greenfield Developers    Kharghar    

## Task 2 - Observe `_score`

Now query the full phrase **"palm beach"**. The surprising result is that the
exact name PJ011 (`Palm Beach`) and the partial name PJ012
(`Palm Beach Residences`) **tie**. Both share the same builder and locality
matches, and the best-field score saturates the small name-length difference.

That tie is the motivation for the rest of the practical: raw BM25 is not a
business ranking. It only knows term statistics, not "exact name is better
than a partial name".

In [4]:
baseline_query = {
    "multi_match": {
        "query": "palm beach",
        "fields": ["name", "builder", "locality", "city", "description"],
    }
}
baseline = search(baseline_query, size=12)
print("total matches:", total_hits(baseline))
print()
score_table(baseline, query_text="palm beach")

base_scores = score_map(baseline)
print()
print("PJ011 (exact name) :", round(base_scores["PJ011"], 4))
print("PJ012 (partial)    :", round(base_scores["PJ012"], 4))
print("PJ015 (builder)    :", round(base_scores["PJ015"], 4))
print("PJ014 (location)   :", round(base_scores["PJ014"], 4))

assert ids_of(baseline)[:2] == ["PJ011", "PJ012"]
assert abs(base_scores["PJ011"] - base_scores["PJ012"]) < 1e-6
assert base_scores["PJ011"] > base_scores["PJ015"] > 0
print()
print("baseline ties the exact and partial names - no business priority yet")

total matches: 14

 # id         score  name                       builder                  locality         signals
----------------------------------------------------------------------------------------------------------------------
 1 PJ011     5.9543  Palm Beach                 Palm Beach Developers    Palm Beach Road  exact-name+builder+location+description
 2 PJ012     5.9543  Palm Beach Residences      Palm Beach Developers    Palm Beach Road  partial-name+builder+location+description
 3 PJ013     5.1873  Palm Beach Enclave         Skyline Builders         Palm Beach Road  partial-name+location+description
 4 PJ014     4.5688  Royal Palm Grove           Palm Estates             Palm Beach Road  location+description
 5 PJ015     3.8665  Crescent Bay               Palm Beach Developers    Nerul            builder+description
 6 PJ010     2.0409  Palm Springs               Palm Estates             Kamothe          -
 7 PJ028     2.0409  Palm Residences            Greenfield Develo

## Task 3 - Compare Scores Across Documents

Three single-field probes show where each score comes from:

- **name** only: PJ011 beats PJ012/PJ013, because the exact name is shorter and
  has a higher BM25 length norm.
- **builder** only: PJ011, PJ012 and PJ015 share the "Palm Beach Developers"
  match; PJ015 has no name match at all.
- **locality** only: PJ011-PJ014 share "Palm Beach Road".

The assignment query then combines the signals with explicit boosts. The
resulting order is exact name → partial names → builder-only → location-only.

In [5]:
probes = {}
for field in ["name", "builder", "locality"]:
    query = {"match": {field: "palm beach"}}
    response = search(query, size=6)
    probes[field] = response
    print(f"match {field:<9} -> {total_hits(response)} hit(s)")
    for hit in hits_of(response)[:4]:
        print(f"  {hit['_id']}  {hit['_score']:.4f}  {hit['_source']['name']}")
    print()

name_scores = score_map(probes["name"])
assert ids_of(probes["name"])[0] == "PJ011"
assert name_scores["PJ011"] > name_scores["PJ012"]
assert abs(name_scores["PJ012"] - name_scores["PJ013"]) < 1e-6
assert "PJ015" in ids_of(probes["builder"])[:3]
assert "PJ014" in ids_of(probes["locality"])

match name      -> 8 hit(s)
  PJ011  5.0007  Palm Beach
  PJ012  4.1881  Palm Beach Residences
  PJ013  4.1881  Palm Beach Enclave
  PJ010  2.0409  Palm Springs



match builder   -> 11 hit(s)
  PJ011  3.8665  Palm Beach
  PJ012  3.8665  Palm Beach Residences
  PJ015  3.8665  Crescent Bay
  PJ003  1.7070  Palm Grove Residences

match locality  -> 4 hit(s)
  PJ011  3.2593  Palm Beach
  PJ012  3.2593  Palm Beach Residences
  PJ013  3.2593  Palm Beach Enclave
  PJ014  3.2593  Royal Palm Grove



In [6]:
EXACT_BOOST = 30
NAME_BOOST = 8
BUILDER_BOOST = 3
LOCATION_BOOST = 1.5
CITY_BOOST = 1
DESCRIPTION_BOOST = 0.5

signal_query = {
    "bool": {
        "should": [
            {"term": {"name.keyword": {"value": "palm beach", "boost": EXACT_BOOST}}},
            {"match_phrase": {"name": {"query": "palm beach", "boost": NAME_BOOST}}},
            {"match_phrase": {"builder": {"query": "palm beach", "boost": BUILDER_BOOST}}},
            {"match_phrase": {"locality": {"query": "palm beach", "boost": LOCATION_BOOST}}},
            {"match_phrase": {"city": {"query": "palm beach", "boost": CITY_BOOST}}},
            {"match_phrase": {"description": {"query": "palm beach", "boost": DESCRIPTION_BOOST}}},
        ],
        "minimum_should_match": 1,
    }
}
signal = search(signal_query, size=10)
print("total matches:", total_hits(signal))
print()
score_table(signal, query_text="palm beach")

signal_ids = ids_of(signal)
signal_scores = score_map(signal)
assert signal_ids[0] == "PJ011"
assert signal_ids.index("PJ011") < signal_ids.index("PJ012") < signal_ids.index("PJ013")
assert signal_ids.index("PJ013") < signal_ids.index("PJ015") < signal_ids.index("PJ014")
assert signal_scores["PJ011"] > signal_scores["PJ012"] > signal_scores["PJ013"]
assert signal_scores["PJ015"] > signal_scores["PJ014"]
print()
print("order: exact name > partial names > builder-only > location-only")

total matches: 5

 # id         score  name                       builder                  locality         signals
----------------------------------------------------------------------------------------------------------------------
 1 PJ011   170.6335  Palm Beach                 Palm Beach Developers    Palm Beach Road  exact-name+builder+location+description
 2 PJ012    52.9702  Palm Beach Residences      Palm Beach Developers    Palm Beach Road  partial-name+builder+location+description
 3 PJ013    40.9871  Palm Beach Enclave         Skyline Builders         Palm Beach Road  partial-name+location+description
 4 PJ015    13.4704  Crescent Bay               Palm Beach Developers    Nerul            builder+description
 5 PJ014     6.7597  Royal Palm Grove           Palm Estates             Palm Beach Road  location+description

order: exact name > partial names > builder-only > location-only


## Task 4 - Explain API

`_explain` returns the score tree for one document and the same query. The
leaves expose the BM25 components:

- `idf, computed as log(1 + (N - n + 0.5) / (n + 0.5))`
- `tf, computed as freq / (freq + k1 * (1 - b + b * dl / avgdl))`
- `dl, length of field` and `avgdl, average length of field`
- `boost` on the clauses that carry a boost

The top-level value equals the document `_score` from the search response.

In [7]:
print("Explain: exact name (PJ011), partial name (PJ012), builder-only (PJ015)")
targets = ["PJ011", "PJ012", "PJ015"]
explained = {}

for project_id in targets:
    explanation = es.explain(
        index=INDEX, id=project_id, query=signal_query
    )["explanation"]
    explained[project_id] = explanation
    rows = []
    explain_walk(explanation, rows)
    print()
    print(f"{project_id} {BY_ID[project_id]['name']!r} "
          f"-> score {explanation['value']:.4f}")
    print(" ", explanation["description"])
    print("  score components:")
    for depth, value, description in rows[:6]:
        print(f"    {value!r:<22} {description}")

for project_id in targets:
    assert abs(explained[project_id]["value"] - signal_scores[project_id]) < 1e-3

rows = []
explain_walk(explained["PJ011"], rows)
descriptions = [description for _, _, description in rows]
assert any(description.startswith("idf") for description in descriptions)
assert any(description.startswith("tf,") for description in descriptions)
assert any(description.startswith("dl,") for description in descriptions)
assert any(description.startswith("avgdl") for description in descriptions)
print()
print("explain value equals the search score; idf/tf/dl/avgdl are visible")

Explain: exact name (PJ011), partial name (PJ012), builder-only (PJ015)



PJ011 'Palm Beach' -> score 170.6335
  sum of:
  score components:
    66.0                   boost
    3.7054088              idf, computed as log(1 + (N - n + 0.5) / (n + 0.5)) from:
    0.45454544             tf, computed as freq / (freq + k1 * (1 - b + b * dl / avgdl)) from:
    1.0                    dl, length of field
    1.0                    avgdl, average length of field
    17.6                   boost



PJ012 'Palm Beach Residences' -> score 52.9702
  sum of:
  score components:
    17.6                   boost
    4.8289185              idf, sum of:
    1.9708077              idf, computed as log(1 + (N - n + 0.5) / (n + 0.5)) from:
    2.858111               idf, computed as log(1 + (N - n + 0.5) / (n + 0.5)) from:
    0.39422208             tf, computed as freq / (freq + k1 * (1 - b + b * dl / avgdl)) from:
    3.0                    dl, length of field

PJ015 'Crescent Bay' -> score 13.4704
  sum of:
  score components:
    6.6000004              boost
    4.526638               idf, sum of:
    1.6685269              idf, computed as log(1 + (N - n + 0.5) / (n + 0.5)) from:
    2.858111               idf, computed as log(1 + (N - n + 0.5) / (n + 0.5)) from:
    0.38826048             tf, computed as freq / (freq + k1 * (1 - b + b * dl / avgdl)) from:
    3.0                    dl, length of field

explain value equals the search score; idf/tf/dl/avgdl are visible


## Task 5 - Boost Project-Name Matches

Raising the `name` field to `name^8` changes what BM25 sees. The exact and
partial names separate (PJ011 above PJ012), name matches move up, and the
location-only document PJ014 drops several places.

A boost multiplies the clause score; it does not filter anything, so the match
set is unchanged.

In [8]:
boosted_query = {
    "multi_match": {
        "query": "palm beach",
        "fields": ["name^8", "builder", "locality", "city", "description"],
    }
}
boosted = search(boosted_query, size=12)
print("before and after name^8:")
print()
print("BEFORE (baseline)")
score_table(baseline, limit=6, query_text="palm beach")
print()
print("AFTER (name^8)")
score_table(boosted, limit=6, query_text="palm beach")

boost_scores = score_map(boosted)
base_ranks = rank_map(baseline)
boost_ranks = rank_map(boosted)
print()
print("PJ011 score:", round(base_scores["PJ011"], 4), "->", round(boost_scores["PJ011"], 4))
print("PJ012 score:", round(base_scores["PJ012"], 4), "->", round(boost_scores["PJ012"], 4))
print("PJ014 rank :", base_ranks["PJ014"], "->", boost_ranks["PJ014"])

assert total_hits(boosted) == total_hits(baseline)
assert ids_of(boosted)[0] == "PJ011"
assert boost_scores["PJ011"] > boost_scores["PJ012"]
assert base_scores["PJ011"] == base_scores["PJ012"]
assert ids_of(baseline) != ids_of(boosted)
assert boost_ranks["PJ014"] > base_ranks["PJ014"]
print()
print("name boost broke the tie and pushed the location-only PJ014 down")

before and after name^8:

BEFORE (baseline)
 # id         score  name                       builder                  locality         signals
----------------------------------------------------------------------------------------------------------------------
 1 PJ011     5.9543  Palm Beach                 Palm Beach Developers    Palm Beach Road  exact-name+builder+location+description
 2 PJ012     5.9543  Palm Beach Residences      Palm Beach Developers    Palm Beach Road  partial-name+builder+location+description
 3 PJ013     5.1873  Palm Beach Enclave         Skyline Builders         Palm Beach Road  partial-name+location+description
 4 PJ014     4.5688  Royal Palm Grove           Palm Estates             Palm Beach Road  location+description
 5 PJ015     3.8665  Crescent Bay               Palm Beach Developers    Nerul            builder+description
 6 PJ010     2.0409  Palm Springs               Palm Estates             Kamothe          -

AFTER (name^8)
 # id         score  nam

## Task 6 - Assign Different Importance to Fields

`multi_match` field boosts decide which field dominates. Query `skyline`:

- `name^10, builder^1` - the name-only project PJ016 (`Skyline`) is first.
- `name^0.1, builder^10` - the top hit is now a project by **Skyline
  Builders**, and PJ016 falls behind the builder matches.

This is field importance, not a filter: every document still matches, but the
ranking is driven by a different field.

In [9]:
name_heavy = {
    "multi_match": {
        "query": "skyline",
        "fields": ["name^10", "builder^1"],
    }
}
builder_heavy = {
    "multi_match": {
        "query": "skyline",
        "fields": ["name^0.1", "builder^10"],
    }
}
name_first = search(name_heavy, size=6)
builder_first = search(builder_heavy, size=6)

print("name-heavy (name^10, builder^1)")
score_table(name_first, query_text="skyline")
print()
print("builder-heavy (name^0.1, builder^10)")
score_table(builder_first, query_text="skyline")

name_ranks = rank_map(name_first)
builder_ranks = rank_map(builder_first)
top_builder = hits_of(builder_first)[0]["_source"]["builder"]
print()
print("name-heavy top:", ids_of(name_first)[0], "| builder-heavy top:", ids_of(builder_first)[0])
print("builder-heavy top builder:", top_builder)
print("PJ016 rank:", name_ranks["PJ016"], "->", builder_ranks.get("PJ016"))
print("PJ018 rank:", name_ranks["PJ018"], "->", builder_ranks["PJ018"])

assert ids_of(name_first)[0] == "PJ016"
assert ids_of(builder_first)[0] != "PJ016"
assert top_builder == "Skyline Builders"
assert builder_ranks["PJ016"] > name_ranks["PJ016"]
assert builder_ranks["PJ018"] <= 4
print()
print("field weights flipped the winner from a name match to a builder match")

name-heavy (name^10, builder^1)
 # id         score  name                       builder                  locality         signals
----------------------------------------------------------------------------------------------------------------------
 1 PJ016    36.7235  Skyline                    Palm Estates             Kharghar         exact-name+description
 2 PJ002    29.5978  Skyline Heights            Skyline Builders         Kharghar         partial-name+builder+description
 3 PJ017    29.5978  Skyline Grand              Skyline Builders         Kharghar         partial-name+builder+description
 4 PJ013     2.6669  Palm Beach Enclave         Skyline Builders         Palm Beach Road  builder+description
 5 PJ018     2.6669  Crescent Bay Vista         Skyline Builders         Nerul            builder+description

builder-heavy (name^0.1, builder^10)
 # id         score  name                       builder                  locality         signals
------------------------------------

## Task 7 - Implement `function_score`

`function_score` lets a business signal modify the BM25 score. This example
combines four functions with `score_mode: sum` and `boost_mode: sum`:

- `field_value_factor(rating, factor=2.0, modifier=log1p)`
- `field_value_factor(views, factor=0.005, modifier=log1p)`
- `gauss(created_at)` - decay with distance from 2025-01-01
- a `filter` on `featured` with weight `8`

The relevance score from Task 3 is the base; popularity can promote a project
(for example the featured, highly viewed PJ014) without letting it beat a true
exact-name match.

In [10]:
function_query = {
    "function_score": {
        "query": signal_query,
        "functions": [
            {
                "field_value_factor": {
                    "field": "rating",
                    "factor": 2.0,
                    "modifier": "log1p",
                    "missing": 3.5,
                }
            },
            {
                "field_value_factor": {
                    "field": "views",
                    "factor": 0.005,
                    "modifier": "log1p",
                    "missing": 0,
                }
            },
            {
                "gauss": {
                    "created_at": {
                        "origin": "2025-01-01",
                        "scale": "365d",
                        "offset": "30d",
                        "decay": 0.5,
                    }
                }
            },
            {"filter": {"term": {"featured": True}}, "weight": 8},
        ],
        "score_mode": "sum",
        "boost_mode": "sum",
    }
}
print("function_score config:")
print(pretty(function_query["function_score"]["functions"]))
print()

scored = search(function_query, size=10)
score_table(scored, query_text="palm beach")

scored_scores = score_map(scored)
scored_ranks = rank_map(scored)
print()
print("PJ014 (featured, 120k views) relevance:",
      round(signal_scores["PJ014"], 4), "->", round(scored_scores["PJ014"], 4))
print("PJ015 (builder-only) relevance      :",
      round(signal_scores["PJ015"], 4), "->", round(scored_scores["PJ015"], 4))
print("PJ014 rank:", signal_ids.index("PJ014") + 1, "->", scored_ranks["PJ014"])

assert ids_of(scored)[0] == "PJ011"
assert scored_ranks["PJ014"] < signal_ids.index("PJ014") + 1
assert scored_scores["PJ014"] > scored_scores["PJ015"]
assert abs(scored_scores["PJ011"] - signal_scores["PJ011"]) < 1e-6 or scored_scores["PJ011"] > signal_scores["PJ011"]
print()
print("popularity promoted PJ014 above the builder-only PJ015; exact name still first")

function_score config:
[
  {
    "field_value_factor": {
      "field": "rating",
      "factor": 2.0,
      "modifier": "log1p",
      "missing": 3.5
    }
  },
  {
    "field_value_factor": {
      "field": "views",
      "factor": 0.005,
      "modifier": "log1p",
      "missing": 0
    }
  },
  {
    "gauss": {
      "created_at": {
        "origin": "2025-01-01",
        "scale": "365d",
        "offset": "30d",
        "decay": 0.5
      }
    }
  },
  {
    "filter": {
      "term": {
        "featured": true
      }
    },
    "weight": 8
  }
]



 # id         score  name                       builder                  locality         signals
----------------------------------------------------------------------------------------------------------------------
 1 PJ011   182.8239  Palm Beach                 Palm Beach Developers    Palm Beach Road  exact-name+builder+location+description
 2 PJ012    57.0530  Palm Beach Residences      Palm Beach Developers    Palm Beach Road  partial-name+builder+location+description
 3 PJ013    44.3851  Palm Beach Enclave         Skyline Builders         Palm Beach Road  partial-name+location+description
 4 PJ014    19.5289  Royal Palm Grove           Palm Estates             Palm Beach Road  location+description
 5 PJ015    17.0140  Crescent Bay               Palm Beach Developers    Nerul            builder+description

PJ014 (featured, 120k views) relevance: 6.7597 -> 19.5289
PJ015 (builder-only) relevance      : 13.4704 -> 17.014
PJ014 rank: 5 -> 4

popularity promoted PJ014 above the build

## Task 8 - Compare Results Before and After Boosting

Four rankings are compared side by side:

1. baseline `multi_match` - no field boost
2. `name^8` - project-name boost
3. assignment relevance query - exact / name / builder / location boosts
4. relevance + `function_score` - business signals added

The exact name PJ011 is first in every business-aware ranking. The bigger
lesson is in the middle of the table: the baseline ties and interleaves
name/builder/location hits, while the assigned boosts produce a stable signal
order, and `function_score` then reorders the lower ranks using popularity.

In [11]:
rankings = {
    "baseline": baseline,
    "name^8": boosted,
    "relevance": signal,
    "relevance+function": scored,
}
order = ["PJ011", "PJ012", "PJ013", "PJ015", "PJ014", "PJ010", "PJ003"]

print(f"{'id':<7}{'name':<27}" + "".join(f"{label:>19}" for label in rankings))
print("-" * 122)
for project_id in order:
    name = BY_ID[project_id]["name"]
    cells_out = []
    for response in rankings.values():
        ranks = rank_map(response)
        cells_out.append(str(ranks.get(project_id, "-")))
    print(f"{project_id:<7}{name:<27}" + "".join(f"{cell:>19}" for cell in cells_out))

final_ranks = rank_map(scored)
base_ranks = rank_map(baseline)
relevance_ranks = rank_map(signal)
print()
print("baseline -> relevance -> relevance+function:")
for project_id in ["PJ011", "PJ012", "PJ013", "PJ015", "PJ014", "PJ010", "PJ003"]:
    print(f"  {project_id}: {base_ranks.get(project_id, '-')} -> "
          f"{relevance_ranks.get(project_id, '-')} -> "
          f"{final_ranks.get(project_id, '-')}")

print()
print("lower-rank swap caused by popularity:")
print("  PJ014:", relevance_ranks["PJ014"], "->", final_ranks["PJ014"])
print("  PJ015:", relevance_ranks["PJ015"], "->", final_ranks["PJ015"])

assert ids_of(scored)[0] == "PJ011"
assert ids_of(signal)[0] == "PJ011"
assert final_ranks["PJ014"] < relevance_ranks["PJ014"]
assert final_ranks["PJ015"] > relevance_ranks["PJ015"]
assert ids_of(baseline) != ids_of(signal)
assert base_ranks.get("PJ010") != final_ranks.get("PJ010")
print()
print("boosted rankings are stable and the exact name wins in all of them")

id     name                                  baseline             name^8          relevance relevance+function
--------------------------------------------------------------------------------------------------------------------------
PJ011  Palm Beach                                   1                  1                  1                  1
PJ012  Palm Beach Residences                        2                  2                  2                  2
PJ013  Palm Beach Enclave                           3                  3                  3                  3
PJ015  Crescent Bay                                 5                  9                  4                  5
PJ014  Royal Palm Grove                             4                  8                  5                  4
PJ010  Palm Springs                                 6                  4                  -                  -
PJ003  Palm Grove Residences                        9                  7                  -         

## Task 9 - Profile API

Adding `profile=True` makes Elasticsearch return the query execution tree:
per-shard timings, the query types that were built, and the per-phase
breakdown (`create_weight`, `build_scorer`, `next_doc`, `advance`, `score`,
`match`, ...).

Setup cost differs between runs - the first profile also pays cold-cache and
warm-up work - so compare the query structure and phase breakdowns rather than
one-off timings. The function query adds a `FunctionScoreQuery` layer above the
phrase/term clauses.

In [12]:
baseline_profile = search(baseline_query, size=3, profile=True)
function_profile = search(function_query, size=3, profile=True)


def report_profile(label, response):
    shards = response["profile"]["shards"]
    tree = shards[0]["searches"][0]["query"][0]
    types = profile_types(tree)
    print(f"{label}")
    print("  shards:", len(shards))
    print("  top query type:", tree["type"])
    print("  time_in_nanos:", tree["time_in_nanos"],
          f"({tree['time_in_nanos'] / 1000:.1f} us)")
    print("  query types:", sorted(set(types)))
    print("  breakdown keys:", ", ".join(sorted(tree["breakdown"])[:8]), "...")
    return types


base_types = report_profile("baseline multi_match", baseline_profile)
print()
func_types = report_profile("relevance + function_score", function_profile)

assert "FunctionScoreQuery" in func_types
assert "FunctionScoreQuery" not in base_types
assert "BooleanQuery" in base_types or "MultiMatchQuery" in base_types
assert len(function_profile["profile"]["shards"]) >= 1
assert function_profile["profile"]["shards"][0]["searches"][0]["query"][0]["breakdown"]
print()
print("the Profile API exposes the function-score layer and the per-phase timings")

baseline multi_match


  shards: 1
  top query type: DisjunctionMaxQuery
  time_in_nanos: 1023833 (1023.8 us)
  query types: ['BooleanQuery', 'DisjunctionMaxQuery', 'TermQuery']
  breakdown keys: advance, advance_count, build_scorer, build_scorer_count, compute_max_score, compute_max_score_count, count_weight, count_weight_count ...

relevance + function_score
  shards: 1
  top query type: FunctionScoreQuery
  time_in_nanos: 1404182 (1404.2 us)
  query types: ['BoostQuery', 'FunctionScoreQuery', 'PhraseQuery', 'TermQuery']
  breakdown keys: advance, advance_count, build_scorer, build_scorer_count, compute_max_score, compute_max_score_count, count_weight, count_weight_count ...

the Profile API exposes the function-score layer and the per-phase timings


## Assignment - Relevance-Based Search System

`relevance_search()` implements the requested rules:

| Rule | Clause | Boost |
|------|--------|-------|
| Exact project name | `term` on `name.keyword` (normalized) | 30 |
| Partial project name | `match_phrase` on `name` | 8 |
| Builder name | `match_phrase` on `builder` | 3 |
| Location | `match_phrase` on `locality` + `city` | 1.5 / 1 |
| Description context | `match_phrase` on `description` | 0.5 |

`minimum_should_match: 1` keeps the query a pure ranking: anything matching
any signal is returned. With `popularity=True` the same `function_score`
layer from Task 7 is applied, but a true exact name still wins.

The documented ranking on **"palm beach"** is:

```
PJ011 Palm Beach             exact name + builder + location   170.63
PJ012 Palm Beach Residences  partial name + builder + location  52.97
PJ013 Palm Beach Enclave     partial name + location            40.99
PJ015 Crescent Bay           builder only                       13.47
PJ014 Royal Palm Grove       location only                       6.76
```

In [13]:
def relevance_search(query_text, popularity=False, size=10):
    exact = {
        "bool": {
            "should": [
                {"term": {"name.keyword": {"value": query_text, "boost": EXACT_BOOST}}},
                {"match_phrase": {"name": {"query": query_text, "boost": NAME_BOOST}}},
                {"match_phrase": {"builder": {"query": query_text, "boost": BUILDER_BOOST}}},
                {"match_phrase": {"locality": {"query": query_text, "boost": LOCATION_BOOST}}},
                {"match_phrase": {"city": {"query": query_text, "boost": CITY_BOOST}}},
                {"match_phrase": {"description": {"query": query_text, "boost": DESCRIPTION_BOOST}}},
            ],
            "minimum_should_match": 1,
        }
    }
    if not popularity:
        query = exact
    else:
        query = {
            "function_score": {
                "query": exact,
                "functions": [
                    {"field_value_factor": {"field": "rating", "factor": 2.0,
                                            "modifier": "log1p", "missing": 3.5}},
                    {"field_value_factor": {"field": "views", "factor": 0.005,
                                            "modifier": "log1p", "missing": 0}},
                    {"gauss": {"created_at": {"origin": "2025-01-01",
                                              "scale": "365d", "offset": "30d",
                                              "decay": 0.5}}},
                    {"filter": {"term": {"featured": True}}, "weight": 8},
                ],
                "score_mode": "sum",
                "boost_mode": "sum",
            }
        }
    return search(query, size=size)


def signal_report(response, query_text, limit=10):
    print(f"{'#':>2} {'id':<6} {'score':>9}  {'name':<27} signals")
    print("-" * 92)
    for rank, hit in enumerate(hits_of(response)[:limit], 1):
        source = hit["_source"]
        print(f"{rank:>2} {hit['_id']:<6} {hit['_score']:>9.4f}  "
              f"{source['name']:<27} {signals(source, query_text)}")


assignment = relevance_search("palm beach", popularity=False)
print("ASSIGNMENT RANKING - query 'palm beach'")
signal_report(assignment, "palm beach")

assignment_ids = ids_of(assignment)
assignment_scores = score_map(assignment)
assert assignment_ids[0] == "PJ011"
assert assignment_ids.index("PJ011") < assignment_ids.index("PJ012")
assert assignment_ids.index("PJ012") < assignment_ids.index("PJ013")
assert assignment_ids.index("PJ013") < assignment_ids.index("PJ015")
assert assignment_ids.index("PJ015") < assignment_ids.index("PJ014")
assert assignment_scores["PJ011"] > assignment_scores["PJ012"] > assignment_scores["PJ013"]

ASSIGNMENT RANKING - query 'palm beach'
 # id         score  name                        signals
--------------------------------------------------------------------------------------------
 1 PJ011   170.6335  Palm Beach                  exact-name+builder+location+description
 2 PJ012    52.9702  Palm Beach Residences       partial-name+builder+location+description
 3 PJ013    40.9871  Palm Beach Enclave          partial-name+location+description
 4 PJ015    13.4704  Crescent Bay                builder+description
 5 PJ014     6.7597  Royal Palm Grove            location+description


In [14]:
popular = relevance_search("palm beach", popularity=True)
print("WITH POPULARITY (function_score)")
signal_report(popular, "palm beach")

popular_ranks = rank_map(popular)
print()
print("PJ014 (featured, 120k views):", rank_map(assignment)["PJ014"], "->", popular_ranks["PJ014"])
assert ids_of(popular)[0] == "PJ011"
assert popular_ranks["PJ014"] < rank_map(assignment)["PJ014"]
print("exact name still wins; popularity changes the lower ranks")

WITH POPULARITY (function_score)
 # id         score  name                        signals
--------------------------------------------------------------------------------------------
 1 PJ011   182.8239  Palm Beach                  exact-name+builder+location+description
 2 PJ012    57.0530  Palm Beach Residences       partial-name+builder+location+description
 3 PJ013    44.3851  Palm Beach Enclave          partial-name+location+description
 4 PJ014    19.5289  Royal Palm Grove            location+description
 5 PJ015    17.0140  Crescent Bay                builder+description

PJ014 (featured, 120k views): 5 -> 4
exact name still wins; popularity changes the lower ranks


In [15]:
probes = [
    ("exact name", "palm beach"),
    ("partial name", "palm beach enclave"),
    ("builder", "palm beach developers"),
    ("location", "palm beach road"),
]

for label, query_text in probes:
    response = relevance_search(query_text, popularity=False, size=5)
    print(f"\n{label}: {query_text!r} -> {total_hits(response)} hit(s)")
    signal_report(response, query_text, limit=5)
    assert total_hits(response) > 0

example = relevance_search("palm beach enclave", popularity=False, size=2)
assert ids_of(example)[0] == "PJ013"
example = relevance_search("palm beach developers", popularity=False, size=3)
assert any("builder" in signals(BY_ID[pid], "palm beach developers")
           for pid in ids_of(example))
example = relevance_search("palm beach road", popularity=False, size=4)
assert all(BY_ID[pid]["locality"] == "Palm Beach Road" for pid in ids_of(example))
print()
print("exact / partial / builder / location probes return the expected signals")


exact name: 'palm beach' -> 5 hit(s)
 # id         score  name                        signals
--------------------------------------------------------------------------------------------
 1 PJ011   170.6335  Palm Beach                  exact-name+builder+location+description
 2 PJ012    52.9702  Palm Beach Residences       partial-name+builder+location+description
 3 PJ013    40.9871  Palm Beach Enclave          partial-name+location+description
 4 PJ015    13.4704  Crescent Bay                builder+description
 5 PJ014     6.7597  Royal Palm Grove            location+description



partial name: 'palm beach enclave' -> 1 hit(s)
 # id         score  name                        signals
--------------------------------------------------------------------------------------------
 1 PJ013   164.4033  Palm Beach Enclave          exact-name+description



builder: 'palm beach developers' -> 3 hit(s)
 # id         score  name                        signals
--------------------------------------------------------------------------------------------
 1 PJ011    16.7955  Palm Beach                  builder+description
 2 PJ012    16.7955  Palm Beach Residences       builder+description
 3 PJ015    16.7955  Crescent Bay                builder+description



location: 'palm beach road' -> 4 hit(s)
 # id         score  name                        signals
--------------------------------------------------------------------------------------------
 1 PJ011     9.7454  Palm Beach                  location+description
 2 PJ012     9.7454  Palm Beach Residences       location+description
 3 PJ013     9.7454  Palm Beach Enclave          location+description
 4 PJ014     9.7454  Royal Palm Grove            location+description



exact / partial / builder / location probes return the expected signals


## BM25 Explained

Elasticsearch scores each query term in each field with BM25 and combines the
clause scores (a `bool` sums its `should` scores).

```
idf(t)   = ln(1 + (N - n(t) + 0.5) / (n(t) + 0.5))

tfNorm   = tf * (k1 + 1)
           --------------------------------
           tf + k1 * (1 - b + b * dl / avgdl)

score(t) = idf(t) * tfNorm * boost
```

- **N** - total documents in the index (60 here).
- **n(t)** - documents containing term `t`. Rare terms have a higher IDF.
- **tf** - occurrences of `t` in the field.
- **dl / avgdl** - field length / average field length. Longer fields are
  penalised by `b`.
- **k1 = 1.2** controls term-frequency saturation; **b = 0.75** controls
  length normalisation. Both are the Elasticsearch defaults used here.
- **boost** multiplies the clause score (for example `name^8` or the exact-name
  `term` boost 30). `function_score` then modifies the final score using
  popularity, not term statistics.

`_explain` (Task 4) prints every one of these factors. The practical shows the
practical consequences: raw BM25 can tie an exact and a partial name (Task 2),
field boosts separate them (Task 5), field weights change the winner (Task 6),
and function scores add business signals without overriding a true exact match
(Tasks 7-8).

In [16]:
SUMMARY = [
    ("Task 1", "Common keyword", "multi_match 'palm'", "14 matches, scores descending"),
    ("Task 2", "Observe _score", "multi_match 'palm beach'", "PJ011 and PJ012 tie at 5.9543"),
    ("Task 3", "Compare scores", "single-field probes + signal query", "exact > partial > builder > location"),
    ("Task 4", "Explain API", "es.explain on PJ011/PJ012/PJ015", "idf/tf/dl/avgdl visible"),
    ("Task 5", "Boost name", "name^8", "PJ011 5.9543 -> 40.0056, PJ014 4 -> 8"),
    ("Task 6", "Field importance", "name^10 vs builder^10 for 'skyline'", "PJ016 -> builder-only top"),
    ("Task 7", "function_score", "rating/views/gauss/featured", "PJ014 promoted above PJ015"),
    ("Task 8", "Before/after", "four rankings", "exact name #1, lower ranks reorder"),
    ("Task 9", "Profile API", "profile=True", "FunctionScoreQuery + timing breakdown"),
    ("Assignment", "Relevance system", "relevance_search('palm beach')", "170.63 / 52.97 / 40.99 / 13.47 / 6.76"),
]

print(f"{'Task':<11} {'Topic':<17} {'Implementation':<38} Evidence")
print("-" * 130)
for task, topic, implementation, evidence in SUMMARY:
    print(f"{task:<11} {topic:<17} {implementation:<38} {evidence}")
print()
print("all relevance tasks completed with score assertions")

Task        Topic             Implementation                         Evidence
----------------------------------------------------------------------------------------------------------------------------------
Task 1      Common keyword    multi_match 'palm'                     14 matches, scores descending
Task 2      Observe _score    multi_match 'palm beach'               PJ011 and PJ012 tie at 5.9543
Task 3      Compare scores    single-field probes + signal query     exact > partial > builder > location
Task 4      Explain API       es.explain on PJ011/PJ012/PJ015        idf/tf/dl/avgdl visible
Task 5      Boost name        name^8                                 PJ011 5.9543 -> 40.0056, PJ014 4 -> 8
Task 6      Field importance  name^10 vs builder^10 for 'skyline'    PJ016 -> builder-only top
Task 7      function_score    rating/views/gauss/featured            PJ014 promoted above PJ015
Task 8      Before/after      four rankings                          exact name #1, lower ranks 

### Wrap-up

- BM25 is field-local: every analyzed field is scored separately and the
  `bool`/`multi_match` combines those scores. A boost changes ranking, never
  the match set.
- Raw BM25 has no business rules; exact and partial names can tie. Put the
  business rules in the query with explicit signals and boosts.
- `function_score` is for signals BM25 cannot see (rating, views, freshness,
  featured status). Keep its weights modest so a true exact match still wins.
- `_explain` is the debugging tool for one document; `_profile` is the
  debugging tool for the whole execution tree. Profile adds overhead, so use
  it during development, not in production traffic.
- `minimum_should_match: 1` on a `should`-only bool turns it into a ranking
  query that still returns every document matching any signal.